In [1]:
import os
import math
import cv2
from pathlib import Path
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from collections import defaultdict
import random

print(torch.__version__)
print(torch.cuda.is_available())

x = torch.randn(4, 4)
x = x.cuda()
print(x.device)

2.2.2+cu121
True
cuda:0


In [2]:
processed_root = Path(
    r"C:\Users\HP\Documents\Codes\Projects\ML\Face Recognition System\data\processed\VGGFace2_160"
)

processed_train = processed_root / "train"
processed_val = processed_root / "val"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

Device: cuda


In [3]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False
        )

        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

        self.conv2 = nn.Conv2d(
            out_channels,
            out_channels,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False
        )

        self.bn2 = nn.BatchNorm2d(out_channels)

        self.shortcut = nn.Sequential()

        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(
                    in_channels,
                    out_channels,
                    kernel_size=1,
                    stride=stride,
                    bias=False
                ),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        identity = self.shortcut(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        out += identity
        out = self.relu(out)

        return out

In [4]:
class FaceRecognitionResNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(3, 2, 1)
        )

        self.layer1 = self._make_layer(64, 64, 2, 1)
        self.layer2 = self._make_layer(64, 128, 2, 2)
        self.layer3 = self._make_layer(128, 256, 2, 2)
        self.layer4 = self._make_layer(256, 512, 2, 2)

        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))

        self.embedding_head = nn.Sequential(
            nn.Linear(512, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),

            nn.Linear(1024, 512),
            nn.BatchNorm1d(512)
        )

    def _make_layer(self, in_channels, out_channels, blocks, stride):
        layers = []
        layers.append(ResidualBlock(in_channels, out_channels, stride))

        for _ in range(1, blocks):
            layers.append(ResidualBlock(out_channels, out_channels))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.stem(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.global_pool(x)
        x = torch.flatten(x, 1)

        embedding = self.embedding_head(x)

        embedding = F.normalize(embedding, p=2, dim=1)

        return embedding

In [5]:
class ArcMarginProduct(nn.Module):
    def __init__(self, in_features, out_features, s=30.0, m=0.5):
        super().__init__()

        self.in_features = in_features
        self.out_features = out_features
        self.s = s
        self.m = m

        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)

        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)

    def forward(self, embeddings, labels):
        cosine = F.linear(
            F.normalize(embeddings),
            F.normalize(self.weight)
        )

        sine = torch.sqrt(torch.clamp(1.0 - cosine**2, min=1e-7))

        phi = cosine * self.cos_m - sine * self.sin_m

        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1), 1.0)

        output = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        output *= self.s

        return output

In [6]:
train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

val_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

In [7]:
all_classes = sorted([
    folder.name
    for folder in processed_train.iterdir()
    if folder.is_dir()
])

class_to_idx = {
    class_name: idx
    for idx, class_name in enumerate(all_classes)
}

num_classes = len(class_to_idx)
print("Train identities:", num_classes)

class FaceDataset(Dataset):
    def __init__(self, root_dir, class_to_idx=None, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.samples = []

        folders = [f for f in self.root_dir.iterdir() if f.is_dir()]

        for folder in folders:
            label = None

            if class_to_idx is not None:
                if folder.name not in class_to_idx:
                    continue
                label = class_to_idx[folder.name]

            image_paths = list(folder.glob("*.jpg"))

            for img_path in image_paths:
                self.samples.append((img_path, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]

        image = cv2.imread(str(img_path))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transform:
            image = self.transform(image)

        return image, label

Train identities: 480


In [8]:
train_dataset = FaceDataset(
    processed_train,
    class_to_idx,
    train_transform
)

batch_size = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

images, labels = next(iter(train_loader))

print(images.shape)
print(labels.shape)

torch.Size([32, 3, 160, 160])
torch.Size([32])


In [9]:
model = FaceRecognitionResNet().to(device)

arcface = ArcMarginProduct(
    in_features=512,
    out_features=num_classes,
    s=30.0,
    m=0.3
).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    list(model.parameters()) + list(arcface.parameters()),
    lr=3e-4,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=20
)

In [10]:
images, labels = next(iter(train_loader))
images = images.to(device)
labels = labels.to(device)

with torch.no_grad():
    embeddings = model(images)
    logits = arcface(embeddings, labels)

print("Embeddings:", embeddings.shape)
print("Logits:", logits.shape)

loss = criterion(logits, labels)
print("Initial loss:", loss.item())

Embeddings: torch.Size([32, 512])
Logits: torch.Size([32, 480])
Initial loss: 16.205120086669922


In [11]:
print("Logits min:", logits.min().item())
print("Logits max:", logits.max().item())
print("Logits mean:", logits.mean().item())

print("Embedding norm mean:", embeddings.norm(dim=1).mean().item())

Logits min: -10.774569511413574
Logits max: 5.4812750816345215
Logits mean: -0.007469802163541317
Embedding norm mean: 1.0


In [12]:
probs = torch.softmax(logits, dim=1)
print("Max probability mean:", probs.max(dim=1)[0].mean().item())

Max probability mean: 0.06222844868898392


In [13]:
def calculate_accuracy(logits, labels):
    preds = torch.argmax(logits, dim=1)
    correct = (preds == labels).sum().item()
    total = labels.size(0)

    return correct / total

In [16]:
def train_one_epoch(model, arcface, loader, criterion, optimizer, device):
    model.train()
    arcface.train()

    running_loss = 0.0
    running_acc = 0.0

    progress_bar = tqdm(loader, desc="Training", leave=False)

    for batch_idx, (images, labels) in enumerate(progress_bar):
        try:
            images = images.to(device)
            labels = labels.to(device)
        except Exception as e:
            print(f"Crash at batch {batch_idx}")
            raise e

        optimizer.zero_grad()

        embeddings = model(images)
        logits = arcface(embeddings, labels)

        loss = criterion(logits, labels)
        loss.backward()

        optimizer.step()

        batch_acc = calculate_accuracy(logits, labels)

        running_loss += loss.item()
        running_acc += batch_acc

        progress_bar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "acc": f"{batch_acc:.4f}"
        })

    epoch_loss = running_loss / len(loader)
    epoch_acc = running_acc / len(loader)

    return epoch_loss, epoch_acc

In [17]:
def train_model(model, arcface, train_loader, criterion, optimizer, scheduler, device, epochs=10):
    history = {
        "train_loss": [],
        "train_acc": []
    }

    best_acc = 0.0

    for epoch in range(epochs):
        print(f"\nEpoch [{epoch+1}/{epochs}]")
        print("-" * 40)

        train_loss, train_acc = train_one_epoch(
            model,
            arcface,
            train_loader,
            criterion,
            optimizer,
            device
        )

        scheduler.step()

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)

        print(f"Train Loss: {train_loss:.4f}")
        print(f"Train Acc : {train_acc:.4f}")

        if train_acc > best_acc:
            best_acc = train_acc

            torch.save({
                "model": model.state_dict(),
                "arcface": arcface.state_dict()
            }, "best_arcface_model.pth")

            print("Saved best model.")

    return history

In [18]:
history = train_model(
    model,
    arcface,
    train_loader,
    criterion,
    optimizer,
    scheduler,
    device,
    epochs=20
)


Epoch [1/20]
----------------------------------------


Train Loss: 12.2056
Train Acc : 0.0074
Saved best model.

Epoch [2/20]
----------------------------------------


Train Loss: 7.5456
Train Acc : 0.1351
Saved best model.

Epoch [3/20]
----------------------------------------


Train Loss: 5.2439
Train Acc : 0.3029
Saved best model.

Epoch [4/20]
----------------------------------------


Train Loss: 4.1016
Train Acc : 0.4151
Saved best model.

Epoch [5/20]
----------------------------------------


...

Train Loss: 0.4150
Train Acc : 0.9186
Saved best model.

Epoch [17/20]
----------------------------------------


Train Loss: 0.3467
Train Acc : 0.9329
Saved best model.

Epoch [18/20]
----------------------------------------


Train Loss: 0.3003
Train Acc : 0.9414
Saved best model.

Epoch [19/20]
----------------------------------------


Train Loss: 0.2694
Train Acc : 0.9481
Saved best model.

Epoch [20/20]
----------------------------------------


Train Loss: 0.2538
Train Acc : 0.9505
Saved best model.


In [19]:
val_identity_to_images = defaultdict(list)

val_folders = [
    folder for folder in processed_val.iterdir()
    if folder.is_dir()
]

for folder in val_folders:
    image_paths = list(folder.glob("*.jpg"))
    
    if len(image_paths) >= 2:
        val_identity_to_images[folder.name] = image_paths

print("Validation identities:", len(val_identity_to_images))

Validation identities: 60


In [20]:
def generate_pairs(identity_to_images, pairs_per_identity=20):
    pairs = []

    identities = list(identity_to_images.keys())

    for identity in identities:
        images = identity_to_images[identity]

        # ---------- Positive pairs ----------
        for _ in range(pairs_per_identity):
            img1, img2 = random.sample(images, 2)
            pairs.append((img1, img2, 1))

        # ---------- Negative pairs ----------
        for _ in range(pairs_per_identity):
            other_identity = random.choice(identities)

            while other_identity == identity:
                other_identity = random.choice(identities)

            img1 = random.choice(images)
            img2 = random.choice(identity_to_images[other_identity])

            pairs.append((img1, img2, 0))

    return pairs

In [21]:
val_pairs = generate_pairs(
    val_identity_to_images,
    pairs_per_identity=20
)

print("Total pairs:", len(val_pairs))
print("Sample pair:", val_pairs[0])

Total pairs: 2400
Sample pair: (WindowsPath('C:/Users/HP/Documents/Codes/Projects/ML/Face Recognition System/data/processed/VGGFace2_160/val/n000001/0027_03.jpg'), WindowsPath('C:/Users/HP/Documents/Codes/Projects/ML/Face Recognition System/data/processed/VGGFace2_160/val/n000001/0154_01.jpg'), 1)


In [23]:
def build_embedding_cache(model, identity_to_images, transform, device):
    model.eval()

    embedding_cache = {}

    with torch.no_grad():
        for identity, image_paths in tqdm(identity_to_images.items(), desc="Caching embeddings"):
            for img_path in image_paths:
                image = cv2.imread(str(img_path))
                image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

                image = transform(image)
                image = image.unsqueeze(0).to(device)

                output = model(image)

                if isinstance(output, tuple):
                    embedding = output[0]
                else:
                    embedding = output
                    
                embedding = embedding.squeeze(0).cpu()

                embedding_cache[str(img_path)] = embedding

    return embedding_cache

In [24]:
embedding_cache = build_embedding_cache(
    model,
    val_identity_to_images,
    val_transform,
    device
)

print("Cached embeddings:", len(embedding_cache))

Caching embeddings: 100%|██████████| 60/60 [03:22<00:00,  3.37s/it]

Cached embeddings: 21221


In [25]:
def compute_similarity(embedding1, embedding2):
    similarity = torch.dot(embedding1, embedding2).item()
    return similarity

In [26]:
pair_scores = []
pair_labels = []

for path1, path2, label in tqdm(val_pairs, desc="Evaluating pairs"):
    emb1 = embedding_cache[str(path1)]
    emb2 = embedding_cache[str(path2)]

    score = compute_similarity(emb1, emb2)

    pair_scores.append(score)
    pair_labels.append(label)

Evaluating pairs: 100%|██████████| 2400/2400 [00:00<00:00, 85719.77it/s]


In [27]:
print("Total scores:", len(pair_scores))
print("Min score:", min(pair_scores))
print("Max score:", max(pair_scores))
print("Mean score:", sum(pair_scores) / len(pair_scores))

Total scores: 2400
Min score: -0.4046216607093811
Max score: 0.9397952556610107
Mean score: 0.17374384814388275


In [28]:
positive_scores = []
negative_scores = []

for score, label in zip(pair_scores, pair_labels):
    if label == 1:
        positive_scores.append(score)
    else:
        negative_scores.append(score)

print("Positive pairs:", len(positive_scores))
print("Negative pairs:", len(negative_scores))

print("\nPositive stats")
print("Min :", min(positive_scores))
print("Max :", max(positive_scores))
print("Mean:", sum(positive_scores) / len(positive_scores))

print("\nNegative stats")
print("Min :", min(negative_scores))
print("Max :", max(negative_scores))
print("Mean:", sum(negative_scores) / len(negative_scores))

Positive pairs: 1200
Negative pairs: 1200

Positive stats
Min : -0.20518198609352112
Max : 0.9397952556610107
Mean: 0.34363711811834946

Negative stats
Min : -0.4046216607093811
Max : 0.5126320719718933
Mean: 0.0038505781694160154


In [29]:
best_acc = 0
best_threshold = None

for threshold in [x / 100 for x in range(-100, 101)]:
    correct = 0

    for score, label in zip(pair_scores, pair_labels):
        pred = 1 if score > threshold else 0

        if pred == label:
            correct += 1

    acc = correct / len(pair_scores)

    if acc > best_acc:
        best_acc = acc
        best_threshold = threshold

print("Best threshold:", best_threshold)
print("Best verification accuracy:", best_acc)

Best threshold: 0.15
Best verification accuracy: 0.8491666666666666


In [30]:
from collections import defaultdict
import random

train_identity_to_images = defaultdict(list)

train_folders = [
    folder for folder in processed_train.iterdir()
    if folder.is_dir()
]

selected_folders = random.sample(train_folders, 20)

for folder in selected_folders:
    image_paths = list(folder.glob("*.jpg"))

    if len(image_paths) >= 2:
        train_identity_to_images[folder.name] = image_paths

print("Selected train identities:", len(train_identity_to_images))

Selected train identities: 20


In [31]:
train_pairs = generate_pairs(
    train_identity_to_images,
    pairs_per_identity=20
)

print("Train pairs:", len(train_pairs))
print("Sample:", train_pairs[0])

Train pairs: 800
Sample: (WindowsPath('C:/Users/HP/Documents/Codes/Projects/ML/Face Recognition System/data/processed/VGGFace2_160/train/n000314/0077_01.jpg'), WindowsPath('C:/Users/HP/Documents/Codes/Projects/ML/Face Recognition System/data/processed/VGGFace2_160/train/n000314/0297_01.jpg'), 1)


In [32]:
train_embedding_cache = build_embedding_cache(
    model,
    train_identity_to_images,
    val_transform,
    device
)

print("Cached train embeddings:", len(train_embedding_cache))

Caching embeddings: 100%|██████████| 20/20 [01:07<00:00,  3.37s/it]

Cached train embeddings: 7491


In [33]:
train_scores = []
train_labels = []

for path1, path2, label in tqdm(train_pairs, desc="Train pair eval"):
    emb1 = train_embedding_cache[str(path1)]
    emb2 = train_embedding_cache[str(path2)]

    score = compute_similarity(emb1, emb2)

    train_scores.append(score)
    train_labels.append(label)

Train pair eval: 100%|██████████| 800/800 [00:00<00:00, 99727.85it/s]


In [34]:
train_positive = []
train_negative = []

for score, label in zip(train_scores, train_labels):
    if label == 1:
        train_positive.append(score)
    else:
        train_negative.append(score)

print("Train positive mean:", sum(train_positive)/len(train_positive))
print("Train negative mean:", sum(train_negative)/len(train_negative))

Train positive mean: 0.7908219644427299
Train negative mean: -0.004072267732117325


In [35]:
checkpoint = {
    "epoch": 20,
    "model_state_dict": model.state_dict(),
    "arcface_state_dict": arcface.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "scheduler_state_dict": scheduler.state_dict()
}

torch.save(checkpoint, "checkpoint_epoch20.pth")
print("Checkpoint saved.")

Checkpoint saved.
